In [1]:
from pathlib import Path
import pandas as pd

cleaned_folder = Path(r"C:\Users\DELL\Desktop\Data Analytics\Project\Cleaned_Data")
october_file = cleaned_folder / "2019-Oct_cleaned.csv"

sample = pd.read_csv(october_file, nrows=5)
sample

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00+00:00,cart,5773203,1487580005134238553,NaN,runail,2.62,463240011,26dd6e6e-4dac-4778-8d2c-92e149dab885
1,2019-10-01 00:00:03+00:00,cart,5773353,1487580005134238553,NaN,runail,2.62,463240011,26dd6e6e-4dac-4778-8d2c-92e149dab885
2,2019-10-01 00:00:07+00:00,cart,5881589,2151191071051219817,NaN,lovely,13.48,429681830,49e8d843-adf3-428b-a2c3-fe8bc6a307c9
3,2019-10-01 00:00:07+00:00,cart,5723490,1487580005134238553,NaN,runail,2.62,463240011,26dd6e6e-4dac-4778-8d2c-92e149dab885
4,2019-10-01 00:00:15+00:00,cart,5881449,1487580013522845895,NaN,lovely,0.56,429681830,49e8d843-adf3-428b-a2c3-fe8bc6a307c9


In [2]:
print("Data types:")
print(sample.dtypes)

print("\nMissing values in these 5 rows:")
print(sample.isna().sum())

Data types:
event_time           str
event_type           str
product_id         int64
category_id        int64
category_code    float64
brand                str
price            float64
user_id            int64
user_session         str
dtype: object

Missing values in these 5 rows:
event_time       0
event_type       0
product_id       0
category_id      0
category_code    5
brand            0
price            0
user_id          0
user_session     0
dtype: int64


In [3]:
sample["event_time"] = pd.to_datetime(sample["event_time"], utc=True)

print("Updated event_time type:", sample["event_time"].dtype)
print(sample["event_time"].head())

Updated event_time type: datetime64[us, UTC]
0   2019-10-01 00:00:00+00:00
1   2019-10-01 00:00:03+00:00
2   2019-10-01 00:00:07+00:00
3   2019-10-01 00:00:07+00:00
4   2019-10-01 00:00:15+00:00
Name: event_time, dtype: datetime64[us, UTC]


In [4]:
print("Event types in the sample:")
print(sample["event_type"].value_counts())

print("\nPrice summary for the sample:")
print(sample["price"].describe())

Event types in the sample:
event_type
cart    5
Name: count, dtype: int64

Price summary for the sample:
count     5.000000
mean      4.380000
std       5.164668
min       0.560000
25%       2.620000
50%       2.620000
75%       2.620000
max      13.480000
Name: price, dtype: float64


In [5]:
event_counts = {}

for chunk in pd.read_csv(october_file, chunksize=500_000):
    counts = chunk["event_type"].value_counts()
    
    for event_type, count in counts.items():
        event_counts[event_type] = event_counts.get(event_type, 0) + count

print(event_counts)

{'view': 1862164, 'cart': 1232385, 'remove_from_cart': 762110, 'purchase': 245624}


In [6]:
files = sorted(cleaned_folder.glob("*_cleaned.csv"))

for file in files:
    missing_counts = {}
    total_rows = 0

    for chunk in pd.read_csv(file, chunksize=500_000):
        total_rows += len(chunk)
        counts = chunk.isna().sum()

        for column, count in counts.items():
            missing_counts[column] = missing_counts.get(column, 0) + count

    print(f"\n{file.name} — {total_rows:,} rows")
    for column, count in missing_counts.items():
        print(f"  {column}: {count:,} missing")


2019-Dec_cleaned.csv — 3,533,286 rows
  event_time: 0 missing
  event_type: 0 missing
  product_id: 0 missing
  category_id: 0 missing
  category_code: 3,474,821 missing
  brand: 1,510,289 missing
  price: 0 missing
  user_id: 0 missing
  user_session: 779 missing

2019-Nov_cleaned.csv — 4,635,837 rows
  event_time: 0 missing
  event_type: 0 missing
  product_id: 0 missing
  category_id: 0 missing
  category_code: 4,560,089 missing
  brand: 1,986,029 missing
  price: 0 missing
  user_id: 0 missing
  user_session: 813 missing

2019-Oct_cleaned.csv — 4,102,283 rows
  event_time: 0 missing
  event_type: 0 missing
  product_id: 0 missing
  category_id: 0 missing
  category_code: 4,034,806 missing
  brand: 1,659,261 missing
  price: 0 missing
  user_id: 0 missing
  user_session: 637 missing

2020-Feb_cleaned.csv — 4,156,682 rows
  event_time: 0 missing
  event_type: 0 missing
  product_id: 0 missing
  category_id: 0 missing
  category_code: 4,079,497 missing
  brand: 1,825,908 missing
  pr

In [7]:
for file in sorted(cleaned_folder.glob("*_cleaned.csv")):
    min_price = None
    max_price = None
    zero_count = 0
    negative_count = 0

    for chunk in pd.read_csv(file, chunksize=500_000):
        chunk_min = chunk["price"].min()
        chunk_max = chunk["price"].max()

        min_price = chunk_min if min_price is None else min(min_price, chunk_min)
        max_price = chunk_max if max_price is None else max(max_price, chunk_max)

        zero_count += (chunk["price"] == 0).sum()
        negative_count += (chunk["price"] < 0).sum()

    print(
        f"{file.name}: min={min_price}, max={max_price}, "
        f"zero prices={zero_count:,}, negative prices={negative_count:,}"
    )

2019-Dec_cleaned.csv: min=-79.37, max=327.78, zero prices=7,589, negative prices=18
2019-Nov_cleaned.csv: min=-79.37, max=327.78, zero prices=11,625, negative prices=17
2019-Oct_cleaned.csv: min=-79.37, max=307.6, zero prices=6,066, negative prices=20
2020-Feb_cleaned.csv: min=-79.37, max=327.78, zero prices=71,254, negative prices=37
2020-Jan_cleaned.csv: min=-79.37, max=327.78, zero prices=7,623, negative prices=39


In [8]:
for file in sorted(cleaned_folder.glob("*_cleaned.csv")):
    negative_events = {}

    for chunk in pd.read_csv(file, chunksize=500_000):
        negative_rows = chunk[chunk["price"] < 0]
        counts = negative_rows["event_type"].value_counts()

        for event_type, count in counts.items():
            negative_events[event_type] = negative_events.get(event_type, 0) + count

    print(f"\n{file.name}")
    print(negative_events)


2019-Dec_cleaned.csv
{'purchase': 18}

2019-Nov_cleaned.csv
{'purchase': 17}

2019-Oct_cleaned.csv
{'purchase': 20}

2020-Feb_cleaned.csv
{'purchase': 35, 'remove_from_cart': 2}

2020-Jan_cleaned.csv
{'purchase': 36, 'view': 2, 'remove_from_cart': 1}


In [9]:
for file in sorted(cleaned_folder.glob("*_cleaned.csv")):
    zero_price_events = {}

    for chunk in pd.read_csv(file, chunksize=500_000):
        zero_rows = chunk[chunk["price"] == 0]
        counts = zero_rows["event_type"].value_counts()

        for event_type, count in counts.items():
            zero_price_events[event_type] = (
                zero_price_events.get(event_type, 0) + count
            )

    print(f"\n{file.name}")
    print(zero_price_events)


2019-Dec_cleaned.csv
{'view': 6067, 'cart': 1207, 'remove_from_cart': 315}

2019-Nov_cleaned.csv
{'view': 8394, 'cart': 2528, 'remove_from_cart': 703}

2019-Oct_cleaned.csv
{'view': 5372, 'cart': 540, 'remove_from_cart': 154}

2020-Feb_cleaned.csv
{'cart': 43423, 'remove_from_cart': 22597, 'view': 5234}

2020-Jan_cleaned.csv
{'view': 5423, 'cart': 1672, 'remove_from_cart': 527, 'purchase': 1}


In [10]:
all_event_types = set()

for file in sorted(cleaned_folder.glob("*_cleaned.csv")):
    for chunk in pd.read_csv(file, usecols=["event_type"], chunksize=500_000):
        all_event_types.update(chunk["event_type"].dropna().unique())

print("Event types found:", sorted(all_event_types))

Event types found: ['cart', 'purchase', 'remove_from_cart', 'view']


In [11]:
session_times = {}

for chunk in pd.read_csv(
    october_file,
    usecols=["user_session", "event_time"],
    chunksize=500_000
):
    chunk["event_time"] = pd.to_datetime(chunk["event_time"], utc=True)

    session_summary = chunk.groupby("user_session")["event_time"].agg(
        ["min", "max"]
    )

    for session_id, row in session_summary.iterrows():
        if session_id not in session_times:
            session_times[session_id] = [row["min"], row["max"]]
        else:
            session_times[session_id][0] = min(
                session_times[session_id][0], row["min"]
            )
            session_times[session_id][1] = max(
                session_times[session_id][1], row["max"]
            )

durations_minutes = [
    (end - start).total_seconds() / 60
    for start, end in session_times.values()
]

pd.Series(durations_minutes).describe()

count    873960.000000
mean         63.110170
std         955.581404
min           0.000000
25%           0.000000
50%           0.000000
75%           1.516667
max       43943.033333
dtype: float64

In [12]:
session_times = {}

for chunk in pd.read_csv(
    october_file,
    usecols=["user_session", "event_time"],
    chunksize=500_000
):
    chunk["event_time"] = pd.to_datetime(chunk["event_time"], utc=True)
    chunk = chunk.dropna(subset=["user_session"])

    session_summary = chunk.groupby("user_session")["event_time"].agg(
        ["min", "max"]
    )

    for session_id, row in session_summary.iterrows():
        if session_id not in session_times:
            session_times[session_id] = [row["min"], row["max"]]
        else:
            session_times[session_id][0] = min(
                session_times[session_id][0], row["min"]
            )
            session_times[session_id][1] = max(
                session_times[session_id][1], row["max"]
            )

durations_minutes = [
    (end - start).total_seconds() / 60
    for start, end in session_times.values()
]

pd.Series(durations_minutes).describe()

count    873960.000000
mean         63.110170
std         955.581404
min           0.000000
25%           0.000000
50%           0.000000
75%           1.516667
max       43943.033333
dtype: float64

In [13]:
october_check = pd.read_csv(
    october_file,
    usecols=["user_session"],
    nrows=10
)

print(october_check["user_session"].isna().sum())
print(october_check["user_session"].head())

0
0    26dd6e6e-4dac-4778-8d2c-92e149dab885
1    26dd6e6e-4dac-4778-8d2c-92e149dab885
2    49e8d843-adf3-428b-a2c3-fe8bc6a307c9
3    26dd6e6e-4dac-4778-8d2c-92e149dab885
4    49e8d843-adf3-428b-a2c3-fe8bc6a307c9
Name: user_session, dtype: str


In [14]:
missing_sessions = 0
unique_sessions = set()

for chunk in pd.read_csv(
    october_file,
    usecols=["user_session"],
    chunksize=500_000
):
    missing_sessions += chunk["user_session"].isna().sum()
    unique_sessions.update(chunk["user_session"].dropna().unique())

print("Missing session rows:", missing_sessions)
print("Distinct nonmissing sessions:", len(unique_sessions))

Missing session rows: 637
Distinct nonmissing sessions: 873960
